# 02 · Bronze → Silver: Limpeza, Tipagem e Correção de Datas (Chamados_Hora)

A tabela `vcredit.silver.tb_chamados_hora` é a versão limpa, tipada e padronizada dos chamados da camada Bronze, preparada para alimentar a camada Gold. Os problemas de qualidade são corrigidos apenas quando há segurança para isso ou sinalizados em colunas `dq_...`, sem exclusão de linhas; somente IDs inválidos e duplicados são desviados para uma tabela de quarentena.

### Inconsistências identificadas na Bronze:
- **Horários:** as 3 colunas de data estão armazenadas como texto (`dd/MM/yyyy às HH:mm:ss`);
- **Aberturas:** 198 chamados com abertura em 2027, mas com início e finalização em 2025/2026;
- **Inícios:** 3.414 chamados sem hora de início;
- **Finalização:** 1 chamado com finalização anterior ao início.

### Objetivo desta etapa:
- Consumir a tabela `vcredit.bronze.tb_chamados_hora`, aplicando limpeza básica de texto.
- Isolar em quarentena os registros com `id_chamado` inválido ou duplicado.
- Converter `id_chamado` para inteiro e os horários para `TimestampType`.
- Corrigir o ano da abertura quando houver divergência evidente em relação ao início e à finalização.
- Calcular os tempos de espera e de atendimento e classificar a situação de cada chamado.
- Sinalizar os problemas de qualidade por meio de colunas `dq_...` e de um status consolidado.
- Garantir idempotência e integridade na gravação em formato **Delta Lake**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Lendo da Bronze: {bronze_schema}")
print(f"Gravando na Silver: {silver_schema}")

---
## Parâmetros de Configuração

### Critérios ajustáveis:
Os valores abaixo concentram os critérios utilizados nas regras de tratamento e podem ser alterados sem modificar o restante do código:
- **`PERIODO_INICIO`** e **`PERIODO_FIM`**: período esperado para a abertura dos chamados;
- **`CORRIGIR_ANO_ABERTURA`**: habilita ou desabilita a correção do ano da abertura;
- **`LIMIAR_DIAS_ANO_DIVERGENTE`** e **`MAX_DURACAO_CORRECAO_SEG`**: condições para aceitar a correção (abertura posterior em 180 dias ou mais e duração resultante de até 24 horas);
- **`MAX_ESPERA_SEG`** e **`MAX_ATENDIMENTO_SEG`**: limites para sinalização de outliers (apenas marcação, sem alteração dos valores);
- **`FORMATOS_DATA`**: formatos de data e hora aceitos na conversão.

In [0]:
PERIODO_INICIO = "2025-01-01 00:00:00"
PERIODO_FIM = "2026-07-31 23:59:59"

CORRIGIR_ANO_ABERTURA = True  # True = corrige o ano da abertura
LIMIAR_DIAS_ANO_DIVERGENTE = 180  # abertura 180+ dias depois do início/fim = erro de ano
MAX_DURACAO_CORRECAO_SEG = 86400  # depois de corrigir, o chamado dura até 24h

MAX_ESPERA_SEG = 3600  # espera acima de 1h = outlier (só marca)
MAX_ATENDIMENTO_SEG = 7200  # atendimento acima de 2h = outlier (só marca)

FORMATOS_DATA = [
    "d/M/yyyy H:mm:ss",
    "d/M/yyyy H:mm",
    "yyyy-M-d H:mm:ss",
    "yyyy-M-d H:mm",
    "yyyy-MM-dd'T'HH:mm:ss",
]

---
## 1. Leitura da Bronze (`vcredit.bronze.tb_chamados_hora`)

A Bronze é lida integralmente em formato texto, e as seguintes padronizações são aplicadas:
- Remoção de espaços em branco (`trim`) em todas as colunas;
- Conversão de textos vazios em `NULL`;
- Preservação dos valores originais em colunas com sufixo `_raw`, mantendo também `ingestion_datetime`.

In [0]:
df_bronze = spark.table(f"{bronze_schema}.tb_chamados_hora")
total_bronze = df_bronze.count()

def limpa_texto(coluna):
    texto = F.trim(F.col(coluna))
    return F.when(texto == "", F.lit(None)).otherwise(texto)

df_base = df_bronze.select(
    limpa_texto("ID_Chamado").alias("id_chamado_raw"),
    limpa_texto("ID_Cliente").alias("id_cliente_raw"),
    limpa_texto("Hora_Abertura_Chamado").alias("abertura_raw"),
    limpa_texto("Hora_Inicio_Atendimento").alias("inicio_raw"),
    limpa_texto("Hora_Finalizacao_Atendimento").alias("fim_raw"),
    F.col("ingestion_datetime")
)

print(f"Linhas na Bronze: {total_bronze}")

---
## 2. IDs inválidos e duplicados

A unicidade de `id_chamado` é assegurada antes de qualquer conversão de tipo:
- **`id_chamado` ausente ou não numérico**: o registro é encaminhado à quarentena com o motivo `ID_CHAMADO_AUSENTE_OU_INVALIDO`;
- **`id_chamado` duplicado**: permanece a linha mais completa (menor quantidade de campos vazios), com desempate pela ingestão mais recente e, em seguida, pelo `hash_linha`. As demais seguem para a quarentena com o motivo `ID_CHAMADO_DUPLICADO_DESCARTADO`;
- **`qtd_linhas_bronze`**: registra quantas linhas existiam na Bronze para cada ID e alimenta a marcação `dq_id_duplicado`.

In [0]:
colunas_raw = ["id_chamado_raw", "id_cliente_raw", "abertura_raw", "inicio_raw", "fim_raw"]

df_base = (
    df_base
    .withColumn("id_chamado_txt", F.regexp_replace(F.col("id_chamado_raw"), r"\.0+$", ""))
    .withColumn("id_valido", F.coalesce(F.col("id_chamado_txt").rlike(r"^\d{1,9}$"), F.lit(False)))
    .withColumn("hash_linha", F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("")) for c in colunas_raw]), 256))
)

df_invalidos = df_base.filter(~F.col("id_valido")).withColumn("motivo", F.lit("ID_CHAMADO_AUSENTE_OU_INVALIDO"))

campos_vazios = sum([F.when(F.col(c).isNull(), 1).otherwise(0) for c in colunas_raw[1:]])

win_chamado = Window.partitionBy("id_chamado_txt").orderBy(
    campos_vazios.asc(), F.col("ingestion_datetime").desc(), F.col("hash_linha").asc()
)
win_contagem = Window.partitionBy("id_chamado_txt")

df_validos = (
    df_base.filter(F.col("id_valido"))
    .withColumn("rn", F.row_number().over(win_chamado))
    .withColumn("qtd_linhas_bronze", F.count("*").over(win_contagem))
)

df_duplicados = df_validos.filter(F.col("rn") > 1).withColumn("motivo", F.lit("ID_CHAMADO_DUPLICADO_DESCARTADO"))
df_unicos = df_validos.filter(F.col("rn") == 1).drop("rn")

def formata_quarentena(df):
    return df.select(
        F.col("id_chamado_raw").alias("id_chamado"),
        F.col("id_cliente_raw").alias("id_cliente"),
        F.col("abertura_raw").alias("hora_abertura_chamado"),
        F.col("inicio_raw").alias("hora_inicio_atendimento"),
        F.col("fim_raw").alias("hora_finalizacao_atendimento"),
        F.col("motivo"),
        F.col("ingestion_datetime")
    )

df_quarentena = formata_quarentena(df_invalidos).unionByName(formata_quarentena(df_duplicados))

---
## 3. Tipagem

- **`id_chamado`**: conversão de `string` para inteiro (`IntegerType`);
- **`id_cliente`**: permanece como texto, com remoção do sufixo `.0` e de separadores (`.`, `-`, `/` e espaços), sendo mantido apenas quando o resultado for numérico;
- **Horários**: convertidos para `TimestampType` a partir dos formatos definidos em `FORMATOS_DATA`, desconsiderando o texto intermediário (ex.: "às") entre data e hora;
- **`veio_abertura`**, **`veio_inicio`** e **`veio_fim`**: indicam se o valor original estava preenchido, permitindo identificar horários ilegíveis.

In [0]:
def converte_data(coluna):
    texto = F.regexp_replace(
        F.col(coluna),
        r"^(\d{1,2}/\d{1,2}/\d{4})\D+(\d{1,2}:\d{2}(:\d{2})?)$",
        "$1 $2"
    )
    return F.coalesce(*[F.try_to_timestamp(texto, F.lit(formato)) for formato in FORMATOS_DATA])

cliente_limpo = F.regexp_replace(F.regexp_replace(F.col("id_cliente_raw"), r"\.0+$", ""), r"[\.\-\s/]", "")

df_tipado = df_unicos.select(
    F.col("id_chamado_txt").cast(IntegerType()).alias("id_chamado"),
    F.when(cliente_limpo.rlike(r"^\d+$"), cliente_limpo).alias("id_cliente"),
    converte_data("abertura_raw").alias("hora_abertura_original"),
    converte_data("inicio_raw").alias("hora_inicio_atendimento"),
    converte_data("fim_raw").alias("hora_finalizacao_atendimento"),
    F.col("abertura_raw").isNotNull().alias("veio_abertura"),
    F.col("inicio_raw").isNotNull().alias("veio_inicio"),
    F.col("fim_raw").isNotNull().alias("veio_fim"),
    F.col("qtd_linhas_bronze"),
    F.col("ingestion_datetime")
)

---
## 4. Correção do ano da abertura

- Este é o **único** horário alterado na camada Silver: a abertura com ano divergente (ex.: 2027) é ajustada para o ano do início do atendimento ou, na ausência deste, da finalização;
- A correção ocorre somente quando a abertura é posterior ao início/finalização em 180 dias ou mais e o chamado resultante tem duração de até 24 horas;
- O valor original é preservado em **`hora_abertura_original`**, e a coluna **`corr_ano_abertura`** indica os registros corrigidos.

In [0]:
df_ano = (
    df_tipado
    .withColumn("ref_ts", F.coalesce("hora_inicio_atendimento", "hora_finalizacao_atendimento"))
    .withColumn("fim_ref", F.coalesce("hora_finalizacao_atendimento", "hora_inicio_atendimento"))
    .withColumn("dif_anos", F.year("ref_ts") - F.year("hora_abertura_original"))
    .withColumn("abertura_try1", F.expr("hora_abertura_original + make_interval(dif_anos, 0, 0, 0, 0, 0, 0)"))
    .withColumn(
        "abertura_ajustada",
        F.when(
            F.col("abertura_try1") > F.col("ref_ts"),
            F.expr("hora_abertura_original + make_interval(dif_anos - 1, 0, 0, 0, 0, 0, 0)")
        ).otherwise(F.col("abertura_try1"))
    )
    .withColumn(
        "corr_ano_abertura",
        F.lit(CORRIGIR_ANO_ABERTURA) & F.coalesce(
            (F.datediff("hora_abertura_original", "ref_ts") >= LIMIAR_DIAS_ANO_DIVERGENTE)
            & (F.col("fim_ref").cast("long") - F.col("abertura_ajustada").cast("long")).between(0, MAX_DURACAO_CORRECAO_SEG),
            F.lit(False)
        )
    )
    .withColumn(
        "hora_abertura_chamado",
        F.when(F.col("corr_ano_abertura"), F.col("abertura_ajustada")).otherwise(F.col("hora_abertura_original"))
    )
)

---
## 5. Tempos e situação do atendimento

- **`espera_seg`** (início − abertura), **`atendimento_seg`** (finalização − início) e **`duracao_total_seg`** (finalização − abertura);
- **`situacao_atendimento`**: `ATENDIDO`, `NAO_ATENDIDO` (sem início, indicando provável abandono), `SEM_FINALIZACAO` ou `SEM_HORARIOS`;
- **`tempo_ate_desistencia_seg`**: duração total dos chamados `NAO_ATENDIDO`, quando maior ou igual a zero;
- **`espera_valida`** e **`atendimento_valido`**: indicam tempo maior ou igual a zero, devendo ser utilizadas no cálculo de médias;
- **`data_abertura`**, **`hora_abertura`** e **`ano_mes`**: atributos derivados do horário de abertura.

In [0]:
df_calc = (
    df_ano
    .withColumn("espera_seg", F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn("atendimento_seg", F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long"))
    .withColumn("duracao_total_seg", F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn(
        "situacao_atendimento",
        F.when(F.col("hora_inicio_atendimento").isNotNull() & F.col("hora_finalizacao_atendimento").isNotNull(), "ATENDIDO")
         .when(F.col("hora_inicio_atendimento").isNull() & F.col("hora_finalizacao_atendimento").isNotNull(), "NAO_ATENDIDO")
         .when(F.col("hora_inicio_atendimento").isNotNull(), "SEM_FINALIZACAO")
         .otherwise("SEM_HORARIOS")
    )
    .withColumn(
        "tempo_ate_desistencia_seg",
        F.when((F.col("situacao_atendimento") == "NAO_ATENDIDO") & (F.col("duracao_total_seg") >= 0), F.col("duracao_total_seg"))
    )
    .withColumn("espera_valida", F.coalesce(F.col("espera_seg") >= 0, F.lit(False)))
    .withColumn("atendimento_valido", F.coalesce(F.col("atendimento_seg") >= 0, F.lit(False)))
    .withColumn("data_abertura", F.to_date("hora_abertura_chamado"))
    .withColumn("hora_abertura", F.hour("hora_abertura_chamado"))
    .withColumn("ano_mes", F.date_format("hora_abertura_chamado", "yyyy-MM"))
)

---
## 6. Marcações de qualidade

- Cada problema identificado é registrado em uma coluna `dq_...` (`true`/`false`), sem exclusão de linhas;
- Finalização anterior ao início **não é alterada**, apenas sinalizada;
- Valores de espera e de atendimento acima dos limites configurados são marcados como outliers, sem alteração do dado;
- **`dq_status`**: classificação consolidada, aplicando-se a primeira condição verdadeira:
  - `INCONSISTENTE`: horário ilegível, abertura nula, ordem cronológica inválida ou abertura fora do período;
  - `INCOMPLETO`: ausência de início, de finalização ou de `id_cliente`;
  - `CORRIGIDO`: ano da abertura ajustado;
  - `OK`: nenhuma das condições anteriores.

In [0]:
df_flags = (
    df_calc
    .withColumn("dq_id_duplicado", F.col("qtd_linhas_bronze") > 1)
    .withColumn("dq_id_cliente_nulo", F.col("id_cliente").isNull())
    .withColumn("dq_abertura_nula", F.col("hora_abertura_chamado").isNull())
    .withColumn("dq_inicio_nulo", F.col("hora_inicio_atendimento").isNull())
    .withColumn("dq_fim_nulo", F.col("hora_finalizacao_atendimento").isNull())
    .withColumn(
        "dq_ts_ilegivel",
        (F.col("veio_abertura") & F.col("hora_abertura_original").isNull())
        | (F.col("veio_inicio") & F.col("hora_inicio_atendimento").isNull())
        | (F.col("veio_fim") & F.col("hora_finalizacao_atendimento").isNull())
    )
    .withColumn("dq_inicio_antes_abertura", F.coalesce(F.col("espera_seg") < 0, F.lit(False)))
    .withColumn("dq_fim_antes_inicio", F.coalesce(F.col("atendimento_seg") < 0, F.lit(False)))
    .withColumn("dq_fim_antes_abertura", F.coalesce(F.col("duracao_total_seg") < 0, F.lit(False)))
    .withColumn(
        "dq_fora_periodo",
        F.coalesce(
            (F.col("hora_abertura_chamado") < F.lit(PERIODO_INICIO).cast("timestamp"))
            | (F.col("hora_abertura_chamado") > F.lit(PERIODO_FIM).cast("timestamp")),
            F.lit(False)
        )
    )
    .withColumn("dq_espera_outlier", F.coalesce(F.col("espera_seg") > MAX_ESPERA_SEG, F.lit(False)))
    .withColumn("dq_atendimento_outlier", F.coalesce(F.col("atendimento_seg") > MAX_ATENDIMENTO_SEG, F.lit(False)))
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_ts_ilegivel") | F.col("dq_abertura_nula") | F.col("dq_inicio_antes_abertura")
            | F.col("dq_fim_antes_inicio") | F.col("dq_fim_antes_abertura") | F.col("dq_fora_periodo"),
            "INCONSISTENTE"
        )
         .when(F.col("dq_inicio_nulo") | F.col("dq_fim_nulo") | F.col("dq_id_cliente_nulo"), "INCOMPLETO")
         .when(F.col("corr_ano_abertura"), "CORRIGIDO")
         .otherwise("OK")
    )
)

---
## 7. Gravação (`vcredit.silver.tb_chamados_hora` e `vcredit.silver.tb_chamados_hora_quarentena`)

- Seleção das colunas finais da Silver, com os horários, tempos, situação e marcações de qualidade;
- Gravação da tabela principal e da quarentena em formato Delta, com `overwrite` e `overwriteSchema`, de modo que a reexecução não gere duplicidade de registros.

In [0]:
df_silver_chamados_hora = df_flags.select(
    "id_chamado",
    "id_cliente",
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento",
    "hora_abertura_original",
    "espera_seg",
    "atendimento_seg",
    "duracao_total_seg",
    "tempo_ate_desistencia_seg",
    "situacao_atendimento",
    "espera_valida",
    "atendimento_valido",
    "data_abertura",
    "hora_abertura",
    "ano_mes",
    "corr_ano_abertura",
    "dq_id_duplicado",
    "dq_id_cliente_nulo",
    "dq_abertura_nula",
    "dq_inicio_nulo",
    "dq_fim_nulo",
    "dq_ts_ilegivel",
    "dq_inicio_antes_abertura",
    "dq_fim_antes_inicio",
    "dq_fim_antes_abertura",
    "dq_fora_periodo",
    "dq_espera_outlier",
    "dq_atendimento_outlier",
    "dq_status",
    "ingestion_datetime"
)

(
    df_silver_chamados_hora.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados_hora")
)

(
    df_quarentena.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados_hora_quarentena")
)

print(f"Tabela {silver_schema}.tb_chamados_hora salva com sucesso!")
print(f"Tabela {silver_schema}.tb_chamados_hora_quarentena salva com sucesso!")
print(f"Total de chamados na Silver: {df_silver_chamados_hora.count()}")
print(f"Total de registros em quarentena: {df_quarentena.count()}")
display(df_silver_chamados_hora)


---
Dessa forma, a tabela `vcredit.silver.tb_chamados_hora` fica tipada e com os problemas tratados ou sinalizados, pronta para consumo pela camada Gold. Os registros desviados constam em `vcredit.silver.tb_chamados_hora_quarentena`.